### Training Data for Multilingual BPE Tokenizer

Language distribution in the training corpus used to build the tokenizer:

| Language/Script | Percentage |
|----------------|------------|
| English        | 60%        |
| Hindi (Deva)   | 10%        |
| Hindi (Latn)   | 10%        |
| Kannada (Knda) | 10%        |
| Kannada (Latn) | 10%        |


In [ ]:
from datasets import load_dataset, load_from_disk
import random
import os

# Load a subset of a dataset and cache it locally
def download_dataset_and_load_subset(dataset_name, name=None, data_dir=None, split="train", num_rows=1000, save_dir="/home/yaseen/hf_datasets"):
    # Create a directory path that includes the dataset name and split
    dataset_path = dataset_name.replace('/', '_')
    save_path = os.path.join(save_dir, f"{dataset_path}_{split}")
    
    try:
        dataset = load_from_disk(save_path)
        print(f"Dataset {dataset_name} loaded from {save_path}..")
    
    except FileNotFoundError:
        # Load the dataset and cache it in the specified directory
        dataset = load_dataset(
            dataset_name, 
            name=name, 
            data_dir=data_dir, 
            split=split
        )
        
        # Ensure the base save directory exists
        os.makedirs(save_dir, exist_ok=True)
        
        # Save the dataset to disk
        dataset.save_to_disk(save_path)
        print(f"Dataset {dataset_name} saved to {save_path}..")
    
    # Select a random subset of the dataset
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    selected_indices = indices[:num_rows]
    return dataset.select(selected_indices)

# Load subsets of each dataset with caching
dataset_en = download_dataset_and_load_subset("HuggingFaceFW/fineweb-edu", name="sample-10BT", num_rows=6000)
dataset_hin_deva = download_dataset_and_load_subset("ai4bharat/sangraha", data_dir="synthetic/hin_Deva")
dataset_hin_latn = download_dataset_and_load_subset("ai4bharat/sangraha", data_dir="synthetic/hin_Latn")
dataset_kan_knda = download_dataset_and_load_subset("ai4bharat/sangraha", data_dir="synthetic/kan_Knda")
dataset_kan_latn = download_dataset_and_load_subset("ai4bharat/sangraha", data_dir="synthetic/kan_Latn")

# Print the first row of each subset to verify
print(dataset_en[0])
print(dataset_hin_deva[0])
print(dataset_hin_latn[0])
print(dataset_kan_knda[0])
print(dataset_kan_latn[0])

In [ ]:
# Concatenate all texts into a single list
all_texts = []

# Collect texts from each dataset
en_texts = [doc["text"].strip().replace("\n", " ") for doc in dataset_en]
hin_deva_texts = [doc["text"].strip().replace("\n", " ") for doc in dataset_hin_deva]
hin_latn_texts = [doc["text"].strip().replace("\n", " ") for doc in dataset_hin_latn]
kan_knda_texts = [doc["text"].strip().replace("\n", " ") for doc in dataset_kan_knda]
kan_latn_texts = [doc["text"].strip().replace("\n", " ") for doc in dataset_kan_latn]

# Add all texts to a single list
all_texts.extend(en_texts)
all_texts.extend(hin_deva_texts)
all_texts.extend(hin_latn_texts)
all_texts.extend(kan_knda_texts)
all_texts.extend(kan_latn_texts)

# Shuffle the combined texts
random.shuffle(all_texts)

print(f"Total number of texts: {len(all_texts)}")
print("\nFirst few texts after random shuffling:")
print(all_texts[:3])

corpus = "\n".join(all_texts)

# Save the combined texts to a file
with open("tok_corpus.txt", "w", encoding="utf-8") as file:
    file.write(corpus)

### Tokenizer Object Training

run train_tokenizer.py

```bash
python train_tokenizer.py
```

### Tokenizer Object Testing

In [ ]:
def test_encoding(tokenizer, text, allowed_special={"<|endoftext|>"}):
    print(f"Text: {text}")
    test_ids = tokenizer.encode(text, verbose=True, allowed_special=allowed_special)
    print("")
    print(f"Unmerged length: {len(text.encode('utf-8'))}")
    print(f"Merged length: {len(test_ids)}")
    print("-"*50)

In [ ]:
tests = [
    "आज तो बहुत थक गया हूँ, ಸ್ವಲ್ಪ विश्रಾಂತಿ ಬೇಕು।",
    "मौसम कितना अच्छा है! ನೀವೂ ಹೊರಗೆ ಬನ್ನಿ, let's enjoy together.",
    "स्वल्पा adjust करो, बैंगलोर का ट्रैफिक ऐसा ही है।",
    "ನೀವು ಚಹಾ ಕುಡಿತೀರಾ? मुझे एक cup चाहिए।",
    "आज का काम पूरा करो, ನಾಳೆ ಎಲ್ಲಿಂದ ಆರಂಭಿಸೋದು ನೋಡಿ।",
    "ಪಾರ್ಟಿ ಹೇಗೆ ಇತ್ತು? मुझे तो बहुत मजा आया!",
    "ನಮ್ಮ ಚೂರು ಸಹನಶೀಲತೆಯನ್ನು ತೋರಿಸಿ, ये थोड़ी देर का मसला है।",
    "ಸಮಯ ನಿಲ್ಲುತ್ತಿಲ್ಲ, जिंदगी में स्वल्पा मज़ा भी जरूरी है।",
    "My name is Yaseen, and I'm a software engineer.<|endoftext|>"
]

for test in tests:
    test_encoding(tokenizer, test)